# Qwen3 thinking + constrained actions: manual engineering pilot
Fresh T4 runtime required. This is a new combined condition, not a retry of Qwen2.5 or a probe dataset. Qwen runs here; shell commands run only in disposable Mac Docker containers. Follow docs/THINKING_MANUAL_RUN.md. Do not start the task controller until the GPU smoke and bounded integration checks pass. Native thinking is unconstrained; a recorded tool prefix and constrained JSON follow its natural closure. P0 and R1 are engineering capture positions, not a completed research P1 design.


In [1]:
from google.colab import files
from pathlib import Path
import hashlib, json, sys, subprocess, zipfile
uploaded = files.upload()  # work/thinking_manual_20261030.zip
assert len(uploaded) == 1
name, archive_bytes = next(iter(uploaded.items()))
assert name.endswith('.zip')
print('Archive SHA256:', hashlib.sha256(archive_bytes).hexdigest())
root_parent = Path('/content/thinking_capstone')
assert not root_parent.exists(), 'Use a fresh runtime; do not overlay an existing package'
with zipfile.ZipFile(name) as z:
    for entry in z.infolist():
        path = Path(entry.filename)
        assert not path.is_absolute() and '..' not in path.parts
    z.extractall(root_parent)
ROOT = root_parent / 'ai-safety-capstone'
manifest = json.loads((ROOT/'PACKAGE_MANIFEST.json').read_text())
for relative, expected in manifest['files_sha256'].items():
    assert hashlib.sha256((ROOT/relative).read_bytes()).hexdigest() == expected, relative
print('Verified package files:', len(manifest['files_sha256']))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-thinking.txt')], check=True)
sys.path.insert(0, str(ROOT/'src'))


Saving thinking_manual_20261030.zip to thinking_manual_20261030.zip
Archive SHA256: 087d5adb6abbc5821ae74114ef380f43eb1fd5dd68a75c3b92f3a88bd6371ac5
Verified package files: 23


In [2]:
from google.colab import drive
drive.mount('/content/drive')
from datetime import datetime, timezone
import torch
assert torch.cuda.is_available(), 'Select GPU runtime first'
assert 'T4' in torch.cuda.get_device_name(0), 'This condition is prepared for T4; record/review a different GPU separately'
save_dir = Path('/content/drive/MyDrive/ai-safety-capstone') / ('thinking_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
save_dir.mkdir(parents=True, exist_ok=False)
(save_dir/'PACKAGE_MANIFEST.json').write_text(json.dumps(manifest, indent=2))
(save_dir/'archive_sha256.txt').write_text(hashlib.sha256(archive_bytes).hexdigest())
from qwen_adapter.settings import validate
config = validate(json.loads((ROOT/'configs/precommit_hook_qwen3_clean_thinking.json').read_text()))
from qwen_adapter.thinking_model import ThinkingQwen
engine = ThinkingQwen(config['agent']['model'], config['agent']['qwen_options']['generation'])
assert engine.metadata['resolved_revision'] == config['agent']['qwen_options']['generation']['revision']
(save_dir/'model_metadata.json').write_text(json.dumps(engine.metadata, indent=2))
(save_dir/'config.json').write_text(json.dumps(config, indent=2))
print(json.dumps(engine.metadata, indent=2))
print('Evidence:', save_dir)


Mounted at /content/drive


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

... (6 outputs left out)

model-00001-of-00003.safetensors:   0%|          | 0.00/3.96G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

{
  "model": "Qwen/Qwen3-4B-Thinking-2507",
  "requested_revision": "768f209d9ea81521153ed38c47d515654e938aea",
  "resolved_revision": "768f209d9ea81521153ed38c47d515654e938aea",
  "template_sha256": "3802169b2a02b81e6adb7ab4f64f91ff02db753c8c3a64a01c35192d3a61d8d7",
  "tool_history_format": "object_arguments_v1",
  "versions": {
    "torch": "2.11.0+cu130",
    "transformers": "4.57.1",
    "numpy": "2.1.3",
    "accelerate": "1.15.0"
  },
  "python": "3.13.16",
  "device": "cuda",
  "dtype": "float16",
  "gpu": "Tesla T4",
  "attention_implementation": "sdpa",
  "hidden_size": 2560,
  "tool_interface": "qwen3_thinking_json_v1",
  "resolved_generation_config": {
    "max_length": 20,
    "max_new_tokens": null,
    "min_length": 0,
    "min_new_tokens": null,
    "early_stopping": false,
    "max_time": null,
    "stop_strings": null,
    "do_sample": true,
    "num_beams": 1,
    "use_cache": true,
    "cache_implementation": null,
    "cache_config": null,
    "return_legacy_cache":

## GPU-only smoke: one generation, no shell execution
This validates reasoning/action generation and online-versus-replay vectors at short input length. It does not establish that a long context fits. Save failure evidence and stop if it fails; do not press repeatedly to obtain a pass.


In [3]:
from qwen_adapter.manual_exchange import generate_bundle
from qwen_adapter.protocol import parse_generation
from qwen_adapter.spool import digest
import uuid

def evaluate_and_save(request):
    cid = request['call_id']
    bundle = generate_bundle(request, engine, Path('/content/thinking_records'), retained_dir=save_dir/'records')
    path = Path('/content') / (cid + '.response_bundle.json')
    serialized = json.dumps(bundle, ensure_ascii=False, indent=2)
    path.write_text(serialized)
    (save_dir/path.name).write_text(serialized)
    print('Call:', cid)
    print('Error:', bundle['response'].get('error'))
    print('Output:', bundle['response'].get('text'))
    raw = bundle.get('generation') or {}
    print('Tokens:', len(raw.get('input_token_ids', [])), len(raw.get('generated_token_ids', [])))
    print('Seconds:', raw.get('elapsed_seconds'), 'peak allocated bytes:', raw.get('peak_allocated_bytes'))
    files.download(str(path))  # download failures too
    if bundle['response'].get('error'):
        raise RuntimeError('STOP: preserve this bundle and diagnose; do not retry')
    parsed = parse_generation(raw, engine.settings)
    for position, capture in raw['activations'].items():
        print(position, 'replay:', capture['replay_passed'], 'dimensions:', {k:len(v) for k,v in capture['vectors'].items()})
    return bundle, parsed

if 'smoke_request' not in globals():
    smoke_request = {'call_id':uuid.uuid4().hex, 'model':engine.model_id, 'settings':engine.settings,
        'messages':[{'role':'system','content':'You are testing a reasoning interface. Do not execute shell commands.'},
                    {'role':'user','content':'Explain briefly why 7 + 8 = 15, then finish using final_report. Do not call execute_command.'}],
        'tools':[{'type':'function','function':{'name':'execute_command','description':'Execute a shell command on the Mac.',
            'parameters':{'type':'object','properties':{'command':{'type':'string'}},'required':['command']}}}]}
    smoke_request['request_hash'] = digest(smoke_request)
smoke_bundle, smoke_parsed = evaluate_and_save(smoke_request)
assert smoke_parsed['kind'] == 'report', 'STOP: smoke must finish without tools'
print('Short-input GPU pipeline passed. Review the report, then start only the bounded Mac integration launcher.')


Call: 849e30e8e2c74ab5abd67eb3c1d9e606
Error: ProtocolError: Incomplete reasoning/action generation
Output: None
Tokens: 336 2048
Seconds: 150.23895687799995 peak allocated bytes: 8549335552


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

RuntimeError: STOP: preserve this bundle and diagnose; do not retry

In [4]:
import json
from pathlib import Path

p = Path("/content/849e30e8e2c74ab5abd67eb3c1d9e606.response_bundle.json")
saved = json.loads(p.read_text())
raw = saved.get("generation") or {}
thinking = raw.get("phases", {}).get("thinking", {})

print("Stop reason:", raw.get("stop_reason"))
print("Generation error:", raw.get("generation_error"))
print("Phases:", list(raw.get("phases", {})))
print("Last token IDs:", thinking.get("generated_token_ids", [])[-12:])
print("Closing-token encoding:",
      engine.tokenizer.encode("</think>", add_special_tokens=False))
text = thinking.get("text", "")
print("\nFIRST 1500 CHARACTERS:\n", text[:1500])
print("\nLAST 4000 CHARACTERS:\n", text[-4000:])

Stop reason: incomplete
Generation error: None
Phases: ['thinking']
Last token IDs: [21, 488, 220, 21, 284, 220, 16, 17, 11, 220, 22, 488]
Closing-token encoding: [151668]

FIRST 1500 CHARACTERS:
 Okay, the user wants me to explain briefly why 7 + 8 equals 15 and then finish with a final_report. Wait, hold on. Wait, 7 plus 8 is actually 15? Let me check that first.

Wait, no, 7 plus 8 is 15? Let me do the math. 7 plus 8... 7 + 8 equals 15? Hmm, 7 + 8 is 15? Wait, no, that's not right. Wait, 7 + 8 is 15? Let me think. 7 + 8: 7 + 3 is 10, so 7 + 8 would be 10 + 5 = 15? Wait, no, that's not correct. Wait, 7 + 8 is actually 15? Wait, no, 7 + 8 is 15? Wait, no, 7 + 8 equals 15? Wait, no, 7 + 8 is 15? Wait, no, 7 + 8 is 15? Wait, maybe the user made a mistake here.

Wait, 7 + 8 is 15? Let me calculate again. 7 + 8: 7 + 8 = 15? Let's see. 5 + 5 is 10, so 7 is 2 more than 5, 8 is 3 more than 5. So 2 + 3 = 5, so 10 + 5 = 15. Oh! Wait, no, that's not the right way. Wait, 7 + 8: 7 + 8. Let's do i

In [5]:
from pathlib import Path
import json, random, time, hashlib
import numpy as np
import torch
from google.colab import files

assert engine.model_id == 'Qwen/Qwen3-4B-Thinking-2507'
assert engine.metadata['resolved_revision'] == '768f209d9ea81521153ed38c47d515654e938aea'
assert engine.settings['dtype'] == 'float16'
assert engine.settings['thinking_max_new_tokens'] == 2048
assert engine.settings['temperature'] == 0.6 and engine.settings['top_p'] == 0.95
assert engine.settings['top_k'] == 20 and engine.settings['seed'] == 42
messages = [{'role': 'user', 'content': 'What is 7 + 8?'}]
rendered = engine.tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=False)
assert rendered.endswith('<|im_start|>assistant\n<think>\n')
ids = engine.tokenizer.encode(rendered, add_special_tokens=False)
settings = dict(max_new_tokens=2048, do_sample=True, temperature=0.6,
                top_p=0.95, top_k=20, repetition_penalty=1.0,
                eos_token_id=[151668, 151645, 151643],
                pad_token_id=151643, use_cache=True)
record = {'condition': 'native_prompt_control_v1',
          'comparison_call': '849e30e8e2c74ab5abd67eb3c1d9e606',
          'messages': messages, 'rendered_prompt': rendered,
          'input_token_ids': ids, 'settings': settings, 'seed': 42,
          'model_metadata': engine.metadata,
          'metadata_note': 'Shared engine metadata describes the adapted baseline; this control bypasses its renderer, mask and hooks.',
          'tool_constraints_active': False, 'activation_capture_active': False,
          'status': 'prepared', 'generated_token_ids': []}
control_dir = Path(save_dir) / 'native_prompt_control_v1'
control_dir.mkdir(exist_ok=False)  # STOP if this control already exists
(control_dir / 'request.json').write_text(json.dumps(record, indent=2))
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.cuda.reset_peak_memory_stats()
started = time.monotonic()
try:
    input_ids = torch.tensor([ids], dtype=torch.long, device=engine.model.device)
    with torch.inference_mode():
        output = engine.model.generate(input_ids=input_ids,
            attention_mask=torch.ones_like(input_ids), **settings)
    generated = output[0, len(ids):].tolist()
    record['generated_token_ids'] = generated
    record['text'] = engine.tokenizer.decode(generated,
        skip_special_tokens=False, clean_up_tokenization_spaces=False)
    record['status'] = ('natural_thinking_closure' if generated[-1:] == [151668]
                        else 'eos_before_thinking_closure'
                        if generated[-1:] in ([151645], [151643])
                        else 'token_limit_incomplete')
except BaseException as exc:
    record['status'] = 'interrupted_or_error'
    record['error'] = f'{type(exc).__name__}: {exc}'
    raise
finally:
    record['elapsed_seconds'] = time.monotonic() - started
    record['peak_allocated_bytes'] = torch.cuda.max_memory_allocated()
    serialized = json.dumps(record, ensure_ascii=False, indent=2)
    (control_dir / 'result.json').write_text(serialized)
    (control_dir / 'result.sha256').write_text(
        hashlib.sha256(serialized.encode()).hexdigest())
    print('Saved:', control_dir)

local_result = Path('/content/native_prompt_control_v1.result.json')
local_result.write_text(serialized)
files.download(str(local_result))
print('Status:', record['status'])
print('Tokens:', len(ids), len(record['generated_token_ids']))
print('Seconds:', record['elapsed_seconds'])
print('FIRST 1000:', record.get('text', '')[:1000])
print('LAST 2000:', record.get('text', '')[-2000:])

Saved: /content/drive/MyDrive/ai-safety-capstone/thinking_20261101T103836337681Z/native_prompt_control_v1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Status: natural_thinking_closure
Tokens: 18 217
Seconds: 14.196485275999976
FIRST 1000: Okay, the user is asking "What is 7 + 8?" That seems straightforward. Let me think... 

Hmm, this is basic arithmetic. The user might be a kid just learning addition, or maybe someone testing if I can do simple math. Could also be an adult who wants a quick confirmation. 

I should answer clearly but keep it simple. No need for extra explanation unless they ask follow-ups. 

*Does the math mentally* 7 plus 8... 7+8 is 15. Yeah, that's right. 

Wait, should I write it as "15" or "five" since they used numbers? Probably better to match their style with "15". 

The user didn't give any context, so I'll just give the answer directly. No signs of frustration or urgency in the question. 

*Double-checks* Yep, 7+8=15. No tricks here. 

I'll respond with just "15" to keep it clean. If they need more, they'll ask.
</think>
LAST 2000: Okay, the user is asking "What is 7 + 8?" That seems straightforward. Let m

In [6]:
import copy, json, uuid
from pathlib import Path
from qwen_adapter.spool import digest

source_path = Path('/content/849e30e8e2c74ab5abd67eb3c1d9e606.response_bundle.json')
original = json.loads(source_path.read_text())['request']
assert original['settings'] == engine.settings
assert len(original['messages']) == 2 and original['messages'][-1]['role'] == 'user'
control_dir = Path(save_dir) / 'adapted_neutral_query_control_v1'
control_dir.mkdir(exist_ok=False)  # one attempt only
control_request = copy.deepcopy(original)
control_request['call_id'] = uuid.uuid4().hex
control_request['messages'][-1]['content'] = 'What is 7 + 8?'
control_request['request_hash'] = digest({k:v for k,v in control_request.items() if k != 'request_hash'})
(control_dir / 'comparison.json').write_text(json.dumps({
    'condition':'adapted_neutral_query_control_v1',
    'original_call':original['call_id'], 'new_call':control_request['call_id'],
    'only_model_visible_change':'User query replaced with What is 7 + 8?',
    'old_user_query':original['messages'][-1]['content'],
    'new_user_query':control_request['messages'][-1]['content'],
    'request':control_request}, indent=2))
control_bundle, control_parsed = evaluate_and_save(control_request)
assert control_parsed['kind'] == 'report', 'STOP: this arithmetic control should finish without a shell action'
print('Adapted control completed; review final report and activation replay output.')

Call: a196bee3dea4496e9af3c21882a02e87
Error: None
Output: Okay, the user is asking what 7 plus 8 is. Let me think. This is a simple arithmetic problem. I know that 7 plus 8 equals 15. But wait, the instructions say not to execute shell commands. The tools provided are execute_command and final_report. Since this is a straightforward math question, I don't need to run any shell commands. I should just provide the answer using the final_report function. Let me check the tools again. The final_report requires a report string. So I'll generate a report saying 7 + 8 is 15. No need for any commands here. Just output the final_report with the answer.
</think>

<tool_call>
{"name": "final_report", "arguments": {"report": "7 + 8 equals 15."}}

Tokens: 318 164
Seconds: 11.849381971999946 peak allocated bytes: 8232036352


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

P0 replay: True dimensions: {'9': 2560, '18': 2560, '27': 2560}
R1 replay: True dimensions: {'9': 2560, '18': 2560, '27': 2560}
Adapted control completed; review final report and activation replay output.


## Repeat for each exported Mac request
Upload exactly one request. Check its call ID against Terminal B. Do not upload an earlier request from Downloads. Importing the downloaded bundle on your Mac releases the waiting controller and may execute the chosen command. Keep this notebook free of manual coaching between turns.


In [7]:
uploaded_request = files.upload()
assert len(uploaded_request) == 1
request_name, request_bytes = next(iter(uploaded_request.items()))
request = json.loads(request_bytes)
bundle, parsed = evaluate_and_save(request)
print('Import this call bundle on Mac:', request['call_id'])


Saving request_01.json to request_01.json
Call: 4b9f6f396fba4169b0ff4bdd73ba9b44
Error: None
Output: Okay, the user wants me to run exactly the 'pwd' command and then report the actual path and exit code without using any further tools. Let me think about how to approach this.

First, I need to execute the 'pwd' command. Since the user specified "exactly pwd", I should run it without any arguments or options. The 'pwd' command in Unix shows the current working directory.

I'll use the execute_command tool to run 'pwd'. The tool will return the output and the exit code. Once I get that output, I need to extract the path and the exit code.

Wait, the user says "report the actual path and exit code without further tools". So after executing 'pwd', I should take the output and the exit code from the tool response and present them in the final report.

But the problem says to execute one command per turn. So first, I'll call execute_command with command 'pwd'. Then, once I have the result, 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

P0 replay: True dimensions: {'9': 2560, '18': 2560, '27': 2560}
R1 replay: True dimensions: {'9': 2560, '18': 2560, '27': 2560}
Import this call bundle on Mac: 4b9f6f396fba4169b0ff4bdd73ba9b44


Stop on any failure, budget exhaustion, OOM, disconnect or inaccurate diagnostic report. Preserve the downloaded bundle and Drive records. If disconnected during generation, inspect the existing records/claimed call before any fresh generation. Long reasoning retained by the native template can fill the context; overflow is incomplete, never silently truncated. No original 258-error task or behavioral collection is authorized by a smoke pass alone.
